In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l5.1 Initialization

Before training, every weight is random, and *how* random matters. Too large and
the loss explodes to NaN; too small and nothing flows. The GPT init (small normal,
std 0.02) starts the model near the uniform baseline `ln(vocab)`, exactly where a
fresh model should be.

In [ ]:
import math
from pocketlm import PocketLM, PocketLMConfig, init_weights

torch.manual_seed(0)
cfg = PocketLMConfig(vocab_size=20, block_size=16, n_layer=2, n_head=2, n_embd=32)
noise = torch.randint(0, 20, (200,))
xb = noise[:16].unsqueeze(0)
yb = noise[1:17].unsqueeze(0)
model = init_weights(PocketLM(cfg), std=0.02)
_, loss = model(xb, yb)
print("init loss:", round(loss.item(), 3), "  ln(vocab):", round(math.log(20), 3))

A sane init means the first loss is finite and near `ln(vocab)`, not a NaN and not
huge. That is the launchpad every later lesson builds on.

In [ ]:
import math
assert torch.isfinite(loss)
assert loss.item() < 2 * math.log(20)